# Bella Italia Chatbot - Midterm Coursework

This notebook implements a food ordering assistant chatbot for Bella Italia restaurant.


## 1. Loading Intents and Menu Data

First, we'll load the intents and menu data from the JSON file.


In [4]:
import json
import os

def load_intents(json_file_path='intents.json'):
    """
    Load intents, menu, and restaurant information from JSON file.
    
    Parameters:
    -----------
    json_file_path : str
        Path to the intents.json file (default: 'intents.json')
    
    Returns:
    --------
    dict
        Dictionary containing restaurant_info, menu, and intents
    """
    try:
        with open(json_file_path, 'r', encoding='utf-8') as file:
            data = json.load(file)
        print(f"Successfully loaded data from {json_file_path}")
        return data
    except FileNotFoundError:
        print(f"Error: File '{json_file_path}' not found.")
        return None
    except json.JSONDecodeError as e:
        print(f"Error: Invalid JSON format in '{json_file_path}': {e}")
        return None

# Load the intents and menu data
data = load_intents('intents.json')

# Display what we loaded
if data:
    print(f"\nRestaurant: {data['restaurant_info']['name']}")
    print(f"Number of intents: {len(data['intents'])}")
    print(f"Menu categories: {list(data['menu'].keys())}")


Successfully loaded data from intents.json

Restaurant: Bella Italia
Number of intents: 3
Menu categories: ['pastas', 'salads', 'drinks']


## 2. Main Loop

The main loop handles user input and terminates when the user types "exit" or "quit".


In [ ]:
def main_loop():
    """
    Main conversation loop for the chatbot.
    Takes user input and terminates when user types "exit" or "quit".
    """
    print("Welcome to Bella Italia Chatbot!")
    print("Type 'exit' or 'quit' to end the conversation.\n")
    
    while True:
        # Get user input
        user_input = input("You: ").strip()
        
        # Check for exit conditions
        if user_input.lower() in ['exit', 'quit']:
            print("Bot: Thank you for visiting Bella Italia! Have a great day!")
            break
        
        # For now, just echo back (will be replaced with actual chatbot logic)
        if user_input:
            print("Bot: (Processing your message...)")
        else:
            print("Bot: Please enter a message or type 'exit' to quit.")

# Run the main loop
# Note: input() doesn't work well in Jupyter notebooks
# Use demo_loop() in the next cell for testing, or run this in a terminal
# main_loop()


Welcome to Bella Italia Chatbot!
Type 'exit' or 'quit' to end the conversation.

Bot: (Processing your message...)
Bot: (Processing your message...)
Bot: Thank you for visiting Bella Italia! Have a great day!


### Note on Input in Jupyter Notebooks

The `input()` function may not work well in Jupyter notebooks. For testing, use the `demo_loop()` function below, or run the chatbot in a terminal.


## 3. Memory Structure

The chatbot needs to remember user information and their order throughout the conversation.


In [2]:
class ChatbotMemory:
    """
    Memory structure to store user information and order details.
    This allows the chatbot to remember context throughout the conversation.
    """
    
    def __init__(self):
        """Initialize empty memory structure."""
        self.user_name = None
        self.cart = []  # List of dictionaries: [{"item": "Spaghetti Carbonara", "quantity": 2, "price": 16.50}, ...]
        self.preferences = []  # List of favorite items (optional)
    
    def set_name(self, name):
        """Store the user's name."""
        self.user_name = name
        print(f"Bot: Nice to meet you, {name}!")
    
    def get_name(self):
        """Get the user's name, or return None if not set."""
        return self.user_name
    
    def add_to_cart(self, item_name, quantity, price):
        """
        Add an item to the cart.
        
        Parameters:
        -----------
        item_name : str
            Name of the item
        quantity : int
            Number of items
        price : float
            Price per item
        """
        self.cart.append({
            "item": item_name,
            "quantity": quantity,
            "price": price
        })
    
    def get_cart(self):
        """Get the current cart."""
        return self.cart
    
    def get_cart_total(self):
        """Calculate total price of items in cart."""
        total = 0.0
        for item in self.cart:
            total += item["quantity"] * item["price"]
        return total
    
    def clear_cart(self):
        """Clear the cart (useful after checkout)."""
        self.cart = []
    
    def add_preference(self, item_name):
        """Add an item to user preferences."""
        if item_name not in self.preferences:
            self.preferences.append(item_name)
    
    def get_preferences(self):
        """Get user preferences."""
        return self.preferences
    
    def reset(self):
        """Reset all memory (useful for testing)."""
        self.user_name = None
        self.cart = []
        self.preferences = []

# Create a global memory instance
memory = ChatbotMemory()

# Test the memory structure
print("=== Testing Memory Structure ===\n")
print(f"Initial state - Name: {memory.get_name()}, Cart: {memory.get_cart()}")
memory.set_name("John")
memory.add_to_cart("Spaghetti Carbonara", 2, 16.50)
memory.add_to_cart("Caesar Salad", 1, 15.00)
print(f"\nAfter adding items:")
print(f"Cart: {memory.get_cart()}")
print(f"Total: £{memory.get_cart_total():.2f}")


=== Testing Memory Structure ===

Initial state - Name: None, Cart: []
Bot: Nice to meet you, John!

After adding items:
Cart: [{'item': 'Spaghetti Carbonara', 'quantity': 2, 'price': 16.5}, {'item': 'Caesar Salad', 'quantity': 1, 'price': 15.0}]
Total: £48.00


## 4. Core Functions Implementation

Functions to process the loaded data and build internal structures for pattern matching and response generation.


In [5]:
def build_pattern_to_intent(data):
    """
    Build a dictionary mapping regex patterns to intent tags.
    
    Parameters:
    -----------
    data : dict
        Dictionary containing intents loaded from JSON
    
    Returns:
    --------
    dict
        Dictionary mapping patterns to intent tags
        Format: {pattern: tag, ...}
    """
    if not data or 'intents' not in data:
        return {}
    
    pattern_to_intent = {}
    
    for intent in data['intents']:
        tag = intent['tag']
        for pattern in intent['patterns']:
            pattern_to_intent[pattern] = tag
    
    return pattern_to_intent

# Test the function
if data:
    pattern_to_intent = build_pattern_to_intent(data)
    print("=== Pattern-to-Intent Dictionary ===\n")
    print(f"Total patterns: {len(pattern_to_intent)}")
    print(f"\nSample mappings:")
    for i, (pattern, tag) in enumerate(list(pattern_to_intent.items())[:5]):
        print(f"  '{pattern}' -> '{tag}'")


=== Pattern-to-Intent Dictionary ===

Total patterns: 22

Sample mappings:
  'hi' -> 'greeting'
  'hello' -> 'greeting'
  'hey' -> 'greeting'
  'good (?:morning|afternoon|evening)' -> 'greeting'
  'greetings' -> 'greeting'


In [6]:
def build_intent_to_response(data):
    """
    Build a dictionary mapping intent tags to their response lists.
    
    Parameters:
    -----------
    data : dict
        Dictionary containing intents loaded from JSON
    
    Returns:
    --------
    dict
        Dictionary mapping intent tags to response lists
        Format: {tag: [response1, response2, ...], ...}
    """
    if not data or 'intents' not in data:
        return {}
    
    intent_to_response = {}
    
    for intent in data['intents']:
        tag = intent['tag']
        intent_to_response[tag] = intent['responses']
    
    return intent_to_response

# Test the function
if data:
    intent_to_response = build_intent_to_response(data)
    print("=== Intent-to-Response Dictionary ===\n")
    print(f"Total intents: {len(intent_to_response)}")
    print(f"\nSample mappings:")
    for tag, responses in intent_to_response.items():
        print(f"  '{tag}': {len(responses)} response(s)")
        print(f"    Example: '{responses[0][:50]}...'")


=== Intent-to-Response Dictionary ===

Total intents: 3

Sample mappings:
  'greeting': 4 response(s)
    Example: 'Hello! Welcome to {restaurant_name}! How can I hel...'
  'name_collection': 4 response(s)
    Example: 'Nice to meet you, {name}! What would you like to o...'
  'menu_inquiry': 4 response(s)
    Example: 'Here's our menu:
{menu_list}
What would you like t...'


In [7]:
def extract_menu_data(data):
    """
    Extract and format menu data from JSON for display.
    
    Parameters:
    -----------
    data : dict
        Dictionary containing menu data loaded from JSON
    
    Returns:
    --------
    dict
        Dictionary with formatted menu data
        Format: {category: [{"name": "...", "price": ..., "description": "..."}, ...], ...}
    """
    if not data or 'menu' not in data:
        return {}
    
    return data['menu']

def format_menu_for_display(menu_data):
    """
    Format menu data into a readable string for display.
    
    Parameters:
    -----------
    menu_data : dict
        Dictionary containing menu items by category
    
    Returns:
    --------
    str
        Formatted menu string ready for display
    """
    if not menu_data:
        return "Menu not available."
    
    menu_text = ""
    
    for category, items in menu_data.items():
        # Capitalize category name
        category_name = category.capitalize()
        menu_text += f"\n**{category_name}:**\n"
        
        for item in items:
            name = item['name']
            price = item['price']
            description = item.get('description', '')
            
            menu_text += f"  • {name} - £{price:.2f}"
            if description:
                menu_text += f"\n    {description}\n"
            else:
                menu_text += "\n"
    
    return menu_text

# Test the functions
if data:
    menu_data = extract_menu_data(data)
    formatted_menu = format_menu_for_display(menu_data)
    print("=== Menu Data Extraction ===\n")
    print(f"Menu categories: {list(menu_data.keys())}")
    print(f"\nFormatted menu preview:")
    print(formatted_menu[:300] + "...")


=== Menu Data Extraction ===

Menu categories: ['pastas', 'salads', 'drinks']

Formatted menu preview:

**Pastas:**
  • Spaghetti Carbonara - £16.50
    Classic Roman pasta with eggs, pancetta, parmesan, and black pepper
  • Penne Arrabbiata - £14.50
    Spicy tomato sauce with garlic, red chili, and fresh basil
  • Fettuccine Alfredo - £17.50
    Rich creamy sauce with parmesan cheese and butter
  •...


## 5. Text Preprocessing

Preprocess user input to improve pattern matching reliability. This includes tokenization, punctuation removal, and lemmatization.


In [11]:
import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Download required NLTK data (run once, then comment out)
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab', quiet=True)

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords', quiet=True)

try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet', quiet=True)

print("NLTK resources ready!")


NLTK resources ready!


In [16]:
def preprocess_input(user_input):
    """
    Preprocess user input to improve pattern matching reliability.
    Always removes stopwords (except important words) and applies lemmatization.
    
    Steps:
    1. Convert to lowercase
    2. Remove punctuation (keep apostrophes for contractions)
    3. Normalize whitespace
    4. Tokenize into words
    5. Remove stop words (keeping important words for pattern matching)
    6. Apply lemmatization to reduce words to root form
    
    Parameters:
    -----------
    user_input : str
        Raw user input string
    
    Returns:
    --------
    str
        Preprocessed input string ready for pattern matching
    """
    if not user_input:
        return ""
    
    # 1. Convert to lowercase
    processed = user_input.lower()
    
    # 2. Remove punctuation (keep apostrophes for contractions and numbers)
    processed = re.sub(r'[^\w\s\'\d]', '', processed)
    
    # 3. Normalize whitespace (remove extra spaces, tabs, newlines)
    processed = re.sub(r'\s+', ' ', processed).strip()
    
    # 4-6. Tokenize and process tokens
    try:
        # Tokenize into words
        tokens = word_tokenize(processed)
        
        # 5. Remove stop words (but keep important words for pattern matching)
        from nltk.corpus import stopwords
        stop_words = set(stopwords.words('english'))
        # Keep important words that are needed for pattern matching
        important_words = {
            'i', 'what', 'how', 'when', 'where', 'who', 'show', 'me', 'my', 'name', 'is',
            'have', 'order', 'menu', 'list', 'display', 'call', 'am', 'food', 'item',
            'option', 'dish', 'serve'
        }
        stop_words = stop_words - important_words
        tokens = [token for token in tokens if token not in stop_words]
        
        # 6. Apply lemmatization (always)
        lemmatizer = WordNetLemmatizer()
        # Try lemmatizing as verb first, then noun
        lemmatized_tokens = []
        for token in tokens:
            # Try verb first (for words like "ordering" -> "order")
            lemmatized = lemmatizer.lemmatize(token, pos='v')
            # If unchanged, try as noun
            if lemmatized == token:
                lemmatized = lemmatizer.lemmatize(token, pos='n')
            lemmatized_tokens.append(lemmatized)
        tokens = lemmatized_tokens
        
        # Join tokens back into string
        processed = ' '.join(tokens)
    except Exception as e:
        # If tokenization fails, return the preprocessed string without lemmatization
        print(f"Warning: Tokenization failed: {e}")
    
    return processed

# Test the preprocessing function
print("=== Testing Preprocessing Function ===\n")

test_inputs = [
    "Hello!",
    "Show me the menu, please.",
    "I'm John",
    "What do you have?",
    "I want to order 2 pizzas",
    "Ordering   multiple   items",
    "HELLO WORLD"
]

print("Original -> Preprocessed:")
for test_input in test_inputs:
    preprocessed = preprocess_input(test_input)
    print(f"  '{test_input}' -> '{preprocessed}'")


=== Testing Preprocessing Function ===

Original -> Preprocessed:
  'Hello!' -> 'hello'
  'Show me the menu, please.' -> 'show me menu please'
  'I'm John' -> 'i 'm john'
  'What do you have?' -> 'what have'
  'I want to order 2 pizzas' -> 'i want order 2 pizza'
  'Ordering   multiple   items' -> 'order multiple item'
  'HELLO WORLD' -> 'hello world'
